# 实际实验结果的 log-likelihood 对比

从本项目 `ll_seismic_model.log` 读取测试集分数，比较 ETAS 与 ST-NKF 的时间、空间和总 LL。没有内置论文分数。同一目录、模型、seed 出现多次时采用日志中最后一次；两种模型必须具有相同的 seed 集合。误差棒为不同 seed 间的样本标准差；只有一个 seed 时不估计标准差，横轴显示 n=1。

改编自 [zhancx-edu/NeuralKernelFunction 的 plot comparison.ipynb](https://github.com/zhancx-edu/NeuralKernelFunction/blob/febc15aefc316cf128ea217bbd60321c88fbaab1/plot%20comparison.ipynb)，适配本仓库 PyTorch 输出。

在仓库根目录启动 Jupyter，先安装 `requirements-plotting.txt`。修改下面的参数单元格后，从上到下执行。结果保存到 `fig/`；本 Notebook 不启动训练。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "run_models.py").is_file():
    raise FileNotFoundError("请在 ST-NKF-pytorch 仓库根目录启动 Jupyter / nbconvert。")

import seaborn as sns

In [ ]:
DATASETS = ["SCEDC_20", "SCEDC_25", "SCEDC_30", "ComCat", "SaltonSea", "SanJac", "WHITE"]
LOG_PATH = ROOT / "ll_seismic_model.log"
# None 使用日志中所有 seed；也可设为 [1, 2, 3]，要求每个目录两种模型都齐全。
SEEDS = None
FIG_DIR = ROOT / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300
SHOW_FIGURES = True

In [ ]:
def save_figure(fig, stem, pdf=False):
    fig.savefig(FIG_DIR / (stem + ".png"), dpi=DPI, bbox_inches="tight")
    if pdf:
        fig.savefig(FIG_DIR / (stem + ".pdf"), bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    print("Saved:", FIG_DIR / (stem + ".png"))

def read_runs():
    if not LOG_PATH.is_file():
        raise FileNotFoundError(f"缺少 {LOG_PATH}；请先运行 ETAS 和 ST-NKF 训练。")
    modes = {
        ("neural", "neural", "neural"): "ST-NKF",
        ("empirical", "empirical", "empirical"): "ETAS",
    }
    rows = []
    for line_no, line in enumerate(LOG_PATH.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        fields = [part.strip() for part in line.split(",")]
        if fields[0] not in DATASETS:
            continue
        if len(fields) < 7:
            raise ValueError(f"{LOG_PATH}:{line_no}: 不完整的实验日志。")
        model = modes.get(tuple(fields[1:4]))
        if model is None:  # 混合消融不属于这张 ETAS / ST-NKF 对比图。
            continue
        seed_fields = [part for part in fields[7:] if part.startswith("seed=")]
        if len(seed_fields) != 1:
            raise ValueError(f"{LOG_PATH}:{line_no}: 需要本项目带 seed=... 的日志。")
        seed = int(seed_fields[0].split("=", 1)[1])
        values = [float(value) for value in fields[4:7]]
        if not np.isfinite(values).all():
            raise ValueError(f"{LOG_PATH}:{line_no}: LL 包含非有限值。")
        rows.append([fields[0], model, seed] + values)
    runs = pd.DataFrame(rows, columns=["Dataset", "Model", "Seed", "Total", "Temporal", "Spatial"])
    if runs.empty:
        raise ValueError("没有找到选定目录的 ETAS / ST-NKF 分数。")
    duplicates = runs.duplicated(["Dataset", "Model", "Seed"], keep="last").sum()
    runs = runs.drop_duplicates(["Dataset", "Model", "Seed"], keep="last")
    if duplicates:
        print(f"采用每个目录 / 模型 / seed 的最后一次记录，忽略 {duplicates} 条较早记录。")
    if SEEDS is not None:
        runs = runs.loc[runs["Seed"].isin(SEEDS)]
    for dataset in DATASETS:
        sets = [set(runs.loc[(runs["Dataset"] == dataset) & (runs["Model"] == model), "Seed"])
                for model in ("ST-NKF", "ETAS")]
        if not sets[0] or sets[0] != sets[1]:
            raise ValueError(f"{dataset}: 两种模型需要相同且非空的 seed 集合，实际为 {sets}。")
        if SEEDS is not None and sets[0] != set(SEEDS):
            raise ValueError(f"{dataset}: 尚未跑齐 SEEDS={SEEDS}。")
    return runs.reset_index(drop=True)


def summarize_runs(runs):
    rows = []
    for dataset in DATASETS:
        for model in ("ST-NKF", "ETAS"):
            group = runs.loc[(runs["Dataset"] == dataset) & (runs["Model"] == model)]
            for metric in ("Temporal", "Spatial", "Total"):
                rows.append([dataset, model, metric, len(group), group[metric].mean(),
                             group[metric].std(ddof=1)])
    return pd.DataFrame(rows, columns=["Dataset", "Model", "Metric", "N", "Mean", "Std"])


def plot_comparison(summary):
    sns.set_theme(style="whitegrid", context="paper", font="DejaVu Sans")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    palette = sns.color_palette("colorblind", 2)
    titles = {"Temporal": "Marginal temporal log-likelihood",
              "Spatial": "Conditional spatial log-likelihood",
              "Total": "Total log-likelihood"}
    x = np.arange(len(DATASETS))
    for ax, metric in zip(axes, ("Temporal", "Spatial", "Total")):
        for index, model in enumerate(("ST-NKF", "ETAS")):
            part = summary.loc[(summary["Metric"] == metric) & (summary["Model"] == model)]
            part = part.set_index("Dataset").loc[DATASETS]
            ax.bar(x + (index - 0.5) * 0.35, part["Mean"].to_numpy(), 0.35,
                   yerr=part["Std"].fillna(0).to_numpy(), capsize=4, label=model,
                   color=palette[index], edgecolor="black", linewidth=0.5)
        counts = summary.loc[(summary["Metric"] == metric) & (summary["Model"] == "ST-NKF")]
        counts = counts.set_index("Dataset").loc[DATASETS, "N"]
        ax.set_xticks(x)
        ax.set_xticklabels([f"{ds}\n(n={n})" for ds, n in zip(DATASETS, counts)],
                           rotation=45, ha="right")
        ax.set_ylabel("Mean test log-likelihood")
        ax.set_title(titles[metric])
        ax.axhline(0, color="black", lw=0.7)
        ax.grid(False, axis="x")
        sns.despine(ax=ax)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
    fig.suptitle("Mean ± sample SD across seeds (n=1: SD not estimated)", fontsize=10)
    fig.tight_layout(rect=[0, 0.06, 1, 0.94])
    save_figure(fig, "log_likelihood_comparison_sci_style", pdf=True)

In [ ]:
runs = read_runs()
summary = summarize_runs(runs)
print(summary.to_string(index=False))
plot_comparison(summary)